In [ ]:
import torch
from transformers import AutoModelForCausalLM
import sys
import os
from dotenv import load_dotenv
from huggingface_hub import snapshot_download

In [ ]:
# Load environment variables
load_dotenv()
deepseek_vl_root = os.getenv("DEEPSEEK_PROJECT_ROOT")

# Ensure the DEEPSEEK_PROJECT_ROOT is correctly set
if not deepseek_vl_root:
    raise ValueError("Error: DEEPSEEK_PROJECT_ROOT is not set. Please check your .env file.")

sys.path.append(deepseek_vl_root)

# Model repository name
model_name = "deepseek-ai/deepseek-vl-7b-chat"
cache_dir = os.path.expanduser("~/.cache/huggingface/transformers")  # Set cache directory

# **Check if model already exists in cache** before downloading
model_path = os.path.join(cache_dir, model_name)
if not os.path.exists(model_path):
    print("🔄 Model not found in cache. Downloading...")
    snapshot_download(repo_id=model_name, cache_dir=cache_dir)
else:
    print(f"✅ Model already cached at: {model_path}")

# Import necessary modules after setting paths
from models.processing_vlm import VLChatProcessor
from models.modeling_vlm import MultiModalityCausalLM
from deepseek_vl.utils.io import load_pil_images


In [ ]:
# Load VLChatProcessor
vl_chat_processor: VLChatProcessor = VLChatProcessor.from_pretrained(
    model_path, cache_dir=cache_dir
)
tokenizer = vl_chat_processor.tokenizer

# Load VL-GPT Model (ensure caching & trust remote code)
vl_gpt: MultiModalityCausalLM = AutoModelForCausalLM.from_pretrained(
    model_path, trust_remote_code=True, cache_dir=cache_dir
)

# Move model to GPU (if available) & set to evaluation mode
device = "cuda" if torch.cuda.is_available() else "cpu"
vl_gpt = vl_gpt.to(torch.bfloat16).to(device).eval()



In [ ]:
# Define the conversation
conversation = [
    {
        "role": "User",
        "content": "<image_placeholder>Describe each stage of this image.",
        "images": ["./a1.png"]
    },
    {
        "role": "Assistant",
        "content": ""
    }
]

In [ ]:
# Load and prepare images for processing
pil_images = load_pil_images(conversation)
prepare_inputs = vl_chat_processor(
    conversations=conversation,
    images=pil_images,
    force_batchify=True
).to(vl_gpt.device)

# Run image encoder to get embeddings
inputs_embeds = vl_gpt.prepare_inputs_embeds(**prepare_inputs)

# Generate response
outputs = vl_gpt.language_model.generate(
    inputs_embeds=inputs_embeds,
    attention_mask=prepare_inputs.attention_mask,
    pad_token_id=tokenizer.eos_token_id,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    max_new_tokens=512,
    do_sample=False,
    use_cache=True
)

# Decode and print the response
answer = tokenizer.decode(outputs[0].cpu().tolist(), skip_special_tokens=True)
print(f"{prepare_inputs['sft_format'][0]}", answer)


In [ ]:
import torch
from transformers import AutoModelForCausalLM
import sys
import os
from dotenv import load_dotenv
from huggingface_hub import snapshot_download

# Load environment variables
load_dotenv()
deepseek_vl_root = os.getenv("DEEPSEEK_PROJECT_ROOT")

# Ensure the DEEPSEEK_PROJECT_ROOT is correctly set
if not deepseek_vl_root:
    raise ValueError("Error: DEEPSEEK_PROJECT_ROOT is not set. Please check your .env file.")

sys.path.append(deepseek_vl_root)

# Model repository name
model_name = "deepseek-ai/deepseek-vl-7b-chat"
cache_dir = os.path.expanduser("~/.cache/huggingface/transformers")

# Construct the expected model path
model_cache_path = os.path.join(cache_dir, f"models--{model_name.replace('/', '--')}")

# **Check if model already exists in cache** before downloading
if not os.path.exists(model_cache_path):
    print("🔄 Model not found in cache. Downloading...")
    snapshot_download(repo_id=model_name, cache_dir=cache_dir)
else:
    print(f"✅ Model already cached at: {model_cache_path}")

# Import necessary modules after setting paths
from models.processing_vlm import VLChatProcessor
from models.modeling_vlm import MultiModalityCausalLM
from deepseek_vl.utils.io import load_pil_images

# Load VLChatProcessor from correct cache path
vl_chat_processor: VLChatProcessor = VLChatProcessor.from_pretrained(
    model_cache_path, cache_dir=cache_dir
)
tokenizer = vl_chat_processor.tokenizer

# Load VL-GPT Model (ensure caching & trust remote code)
vl_gpt: MultiModalityCausalLM = AutoModelForCausalLM.from_pretrained(
    model_cache_path, trust_remote_code=True, cache_dir=cache_dir
)

# Move model to GPU (if available) & set to evaluation mode
device = "cuda" if torch.cuda.is_available() else "cpu"
vl_gpt = vl_gpt.to(torch.bfloat16).to(device).eval()

# Define the conversation
conversation = [
    {
        "role": "User",
        "content": "<image_placeholder>Describe each stage of this image.",
        "images": ["./a1.png"]
    },
    {
        "role": "Assistant",
        "content": ""
    }
]

# Load and prepare images for processing
pil_images = load_pil_images(conversation)
prepare_inputs = vl_chat_processor(
    conversations=conversation,
    images=pil_images,
    force_batchify=True
).to(vl_gpt.device)

# Run image encoder to get embeddings
inputs_embeds = vl_gpt.prepare_inputs_embeds(**prepare_inputs)

# Generate response
outputs = vl_gpt.language_model.generate(
    inputs_embeds=inputs_embeds,
    attention_mask=prepare_inputs.attention_mask,
    pad_token_id=tokenizer.eos_token_id,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    max_new_tokens=512,
    do_sample=False,
    use_cache=True
)

# Decode and print the response
answer = tokenizer.decode(outputs[0].cpu().tolist(), skip_special_tokens=True)
print(f"{prepare_inputs['sft_format'][0]}", answer)


In [ ]:
from huggingface_hub import snapshot_download

model_name = "deepseek-ai/deepseek-vl-7b-chat"
cache_dir = os.path.expanduser("~/.cache/huggingface/transformers")

# Force fresh download to avoid missing files
snapshot_download(repo_id=model_name, cache_dir=cache_dir, force_download=True)


In [ ]:
import os
import torch
from transformers import AutoModelForCausalLM
from deepseek_vl.models.processing_vlm import VLChatProcessor

cache_dir = os.path.expanduser("~/.cache/huggingface/transformers")
model_name = "deepseek-ai/deepseek-vl-7b-chat"

# Load VLChatProcessor correctly
vl_chat_processor = VLChatProcessor.from_pretrained(model_name, cache_dir=cache_dir)
tokenizer = vl_chat_processor.tokenizer

# Load the language model
vl_gpt = AutoModelForCausalLM.from_pretrained(
    model_name, trust_remote_code=True, cache_dir=cache_dir
)

# Move model to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
vl_gpt = vl_gpt.to(torch.bfloat16).to(device).eval()


In [ ]:
import os
from huggingface_hub import snapshot_download

# Model repository name on Hugging Face
model_name = "deepseek-ai/deepseek-vl-7b-chat"
cache_dir = os.path.expanduser("~/.cache/huggingface/transformers")

# Force a full download and extraction of the model
model_path = snapshot_download(
    repo_id=model_name,
    cache_dir=cache_dir,
    force_download=True  # ensures all files are freshly downloaded and extracted
)

print(f"Model fully downloaded and extracted to: {model_path}")


In [ ]:
import os
import sys
import torch
from transformers import AutoModelForCausalLM
from dotenv import load_dotenv

# Load environment variables and add project root to sys.path
load_dotenv()
deepseek_vl_root = os.getenv("DEEPSEEK_PROJECT_ROOT")
if not deepseek_vl_root:
    raise ValueError("DEEPSEEK_PROJECT_ROOT is not set in your .env file.")
sys.path.append(deepseek_vl_root)

# Define the cache and model name
cache_dir = os.path.expanduser("~/.cache/huggingface/transformers")
model_name = "deepseek-ai/deepseek-vl-7b-chat"

# Construct the model cache path that Hugging Face creates.
# Hugging Face converts '/' to '--' in folder names.
#model_cache_path = os.path.join(cache_dir, f"models--{model_name.replace('/', '--')}")

print("Loading model from cache path:", model_path)

# Import DeepSeek VL modules after adjusting sys.path
from models.processing_vlm import VLChatProcessor
from models.modeling_vlm import MultiModalityCausalLM
from deepseek_vl.utils.io import load_pil_images

# Load VLChatProcessor from the cached model path
vl_chat_processor = VLChatProcessor.from_pretrained(model_cache_path, cache_dir=cache_dir)
tokenizer = vl_chat_processor.tokenizer

# Load the language model using AutoModelForCausalLM
vl_gpt = AutoModelForCausalLM.from_pretrained(
    model_path,
    trust_remote_code=True,
    cache_dir=cache_dir
)

# Move model to GPU if available and set to evaluation mode.
device = "cuda" if torch.cuda.is_available() else "cpu"
vl_gpt = vl_gpt.to(torch.bfloat16).to(device).eval()

# Define a sample conversation
conversation = [
    {
        "role": "User",
        "content": "<image_placeholder>Describe each stage of this image.",
        "images": ["./a1.png"]  # adjust path to your image
    },
    {
        "role": "Assistant",
        "content": ""
    }
]

# Load and prepare images for processing
pil_images = load_pil_images(conversation)
prepare_inputs = vl_chat_processor(
    conversations=conversation,
    images=pil_images,
    force_batchify=True
).to(vl_gpt.device)

# Run the image encoder to get input embeddings
inputs_embeds = vl_gpt.prepare_inputs_embeds(**prepare_inputs)

# Generate the response
outputs = vl_gpt.language_model.generate(
    inputs_embeds=inputs_embeds,
    attention_mask=prepare_inputs.attention_mask,
    pad_token_id=tokenizer.eos_token_id,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    max_new_tokens=512,
    do_sample=False,
    use_cache=True
)

# Decode and print the response
answer = tokenizer.decode(outputs[0].cpu().tolist(), skip_special_tokens=True)
print(f"{prepare_inputs['sft_format'][0]}", answer)


In [ ]:
import os
import sys
import torch
from transformers import AutoModelForCausalLM
from dotenv import load_dotenv
from huggingface_hub import snapshot_download

# --- Environment & Path Setup ---
load_dotenv()
deepseek_vl_root = os.getenv("DEEPSEEK_PROJECT_ROOT")
if not deepseek_vl_root:
    raise ValueError("DEEPSEEK_PROJECT_ROOT is not set in your .env file.")
sys.path.append(deepseek_vl_root)

# --- Model Repository & Cache Settings ---
model_name = "deepseek-ai/deepseek-vl-7b-chat"
cache_dir = os.path.expanduser("~/.cache/huggingface/transformers")

# Force a full download to ensure the cache is complete
snapshot_download(repo_id=model_name, cache_dir=cache_dir, force_download=True)
print(f"Model cached in: {cache_dir}")

# --- Import DeepSeek VL Modules ---
from models.processing_vlm import VLChatProcessor
from models.modeling_vlm import MultiModalityCausalLM
from deepseek_vl.utils.io import load_pil_images

# --- Load Processor and Model Using the Repository ID ---
# Note: We pass 'model_name' (the repo id) rather than a local snapshot path.
vl_chat_processor = VLChatProcessor.from_pretrained(model_name, cache_dir=cache_dir)
tokenizer = vl_chat_processor.tokenizer

vl_gpt = AutoModelForCausalLM.from_pretrained(
    model_name, trust_remote_code=True, cache_dir=cache_dir
)

# --- Move Model to Device ---
device = "cuda" if torch.cuda.is_available() else "cpu"
vl_gpt = vl_gpt.to(torch.bfloat16).to(device).eval()

# --- Define a Sample Conversation ---
conversation = [
    {
        "role": "User",
        "content": "<image_placeholder>Describe each stage of this image.",
        "images": ["./a1.png"]  # Adjust this path to your image
    },
    {
        "role": "Assistant",
        "content": ""
    }
]

# --- Process Images and Prepare Inputs ---
pil_images = load_pil_images(conversation)
prepare_inputs = vl_chat_processor(
    conversations=conversation,
    images=pil_images,
    force_batchify=True
).to(device)

# --- Run the Model ---
inputs_embeds = vl_gpt.prepare_inputs_embeds(**prepare_inputs)
outputs = vl_gpt.language_model.generate(
    inputs_embeds=inputs_embeds,
    attention_mask=prepare_inputs.attention_mask,
    pad_token_id=tokenizer.eos_token_id,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    max_new_tokens=512,
    do_sample=False,
    use_cache=True
)

# --- Decode and Print the Response ---
answer = tokenizer.decode(outputs[0].cpu().tolist(), skip_special_tokens=True)
print(f"{prepare_inputs['sft_format'][0]}", answer)
